# VITAL demo: score ClinVar variants in 2 minutes

This notebook scores cached ClinVar VCV records with VITAL and prints a clean review summary.

You get a VITAL score, band, key signals, and component breakdown. The demo uses cached score tables, so it does not call ClinVar or gnomAD APIs.

In [ ]:
# Setup: clone the repo if this notebook is running in a fresh Colab session.
from pathlib import Path

REPO_URL = "https://github.com/anerecye/New-project.git"

if not Path("run_vital.py").exists():
    if not Path("vital").exists():
        !git clone --depth 1 -q {REPO_URL} vital
    %cd vital
else:
    print("Already in VITAL repo:", Path.cwd())

try:
    import pandas as pd
except ImportError:
    !pip -q install pandas
    import pandas as pd

print("Ready. This demo uses cached VITAL scores; no API calls.")

## Main demo

Edit `VCV_ID` and run the cell.

Try: `VCV000440850` (SCN5A), `VCV001325231` (TRDN), or `VCV004535537` (KCNH2).

In [1]:
VCV_ID = "VCV000440850"  # Edit this line and run

!python run_vital.py --vcv $VCV_ID

VITAL cached demo result
Query:        VCV000440850
ClinVar ID:   VCV000440850
Gene:         SCN5A
Variant:      NM_000335.4(SCN5A):c.[3919C>T;694G>A]

VITAL score:  96.1
Band:         RED (reclassification-priority)
Red flag:     True

Frequency:
  Global AF / AC: 1.465e-04 / 214
  Popmax AF / AC: 0.00568 / 190 (afr)

Review:
  ClinVar review: no assertion criteria provided
  Review strength: weak_or_no_assertion

Key signals:
- high population frequency signal (max AF 0.00568)
- AC-supported evidence (qualifying AC 214)
- population-specific enrichment (popmax/global ratio 38.7; popmax=afr)
- weak or single-submitter ClinVar review support
- manual expert re-review recommended; not automatic benign classification

Component scores:
  AF pressure               45.0/45
  AC reliability            20.0/20
  Popmax enrichment          7.9/10
  Variant type tension       0.0/6
  Technical detectability    6.5/8
  Gene constraint            6.6/10
  Review fragility          10.0/10


## How to read the result

- `RED`: immediate expert re-review queue. This is not an automatic benign classification.
- `ORANGE` / `YELLOW`: frequency tension worth watching or batch review.
- `GRAY`: no usable exact frequency evidence. Do not treat as AF=0.

VITAL is a prioritization layer. Final interpretation still needs phenotype, inheritance, segregation, functional evidence, and lab policy.

## Batch mode

Use a CSV with a `vcv`, `clinvar_id`, or `variation_id` column.

In [ ]:
!python run_vital.py --input data/examples/sample_variants.csv --output vital_batch_results.csv

In [ ]:
pd.read_csv("vital_batch_results.csv")

## Demo dataset

Want to browse instead of entering a VCV? Start with the cached top suspicious table.

In [ ]:
cols = [
    "priority_rank", "gene", "clinvar_id", "title", "global_af", "global_ac",
    "popmax_af", "popmax_ac", "popmax_population", "vital_score", "vital_band", "vital_red_flag",
]

pd.read_csv("data/processed/vital_top_suspicious.csv")[cols].head(5)